# Chapter 6 — Evaluating a model and measuring uncertainty · Notebook to complete

*Maths for AI, step by step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch06_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to modify are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without raising an error. The solution is in `ch06_corrige.ipynb`, on the same companion page.

**Contents.** Data · The calculations of lessons 6.1 to 6.6 · Exercises 6.1 to 6.6

**The protocol, throughout the notebook:** fit on the **training** set, choose on the **validation** set, evaluate **only once** on the **test** set. For an A/B test: compare two **randomly assigned** groups, and give the difference **with its confidence interval**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book colours: one colour per set
LOT_E, LOT_V, LOT_T = '#2e7d32', '#b84d00', '#1565c0'   # training (green), validation (dark orange), test (blue)
SPAM, LEGITIME = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, ECART, SEUIL = '#757575', '#00897b', '#c2185b', '#8e24aa'   # A/B test: A grey, B teal


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def perte_moyenne(etiquettes, X, w, b):
    """Mean log loss (equations 5.6 and 5.7)."""
    p = np.clip(sigmoide(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(etiquettes, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


NOMS_CARAC = ['liens', 'mots_suspects', 'longueur', 'majuscules', 'heure', 'pieces_jointes', 'taille', 'destinataires']


def fabriquer_messages(graine=77):
    """Fictitious messages (exercise 6.4 and lesson 6.4): 40 groups, spam campaigns (even groups) or
    legitimate senders (odd groups). The messages of a group are almost identical. Only the first two
    features depend on the label; the eight are in very different units."""
    rng = np.random.default_rng(graine)
    unites = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    decalages = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    lignes = []
    for g in range(40):
        spam = g % 2 == 0
        taille = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(taille):
            lignes.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * unites + decalages))
    return pd.DataFrame(lignes, columns=['groupe', 'spam'] + NOMS_CARAC)


def ajuster(X, y, lam, l1=False, pas=0.1, n_pas=3000):
    """Gradient descent on the penalised loss: L2 (equation 6.2) or L1 (equation 6.3). b is not penalised."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_pas):
        ecart = sigmoide(X @ w + b) - y
        gw, gb = X.T @ ecart / len(y), ecart.mean()
        if l1:   # gradient step, then move each weight towards 0 by at most pas × λ (without changing its sign)
            w = w - pas * gw
            w = np.sign(w) * np.maximum(np.abs(w) - pas * lam, 0)
        else:    # the derivative of λ × sum of the w² is 2λw
            w = w - pas * (gw + 2 * lam * w)
        b -= pas * gb
    return w, b

## Data

The identifiers D6.1 to D6.5, D6.E1, D6.E2, D6.E3, D6.E5 and D5.1 are those of the book and of the CSV files.

Variable and column names in the code are kept in French: `lot` (set: `entrainement` training, `validation`, `test`), `seuil` (threshold), `fn` / `fp` (false negatives / false positives), `coût` (cost), `pli` (fold), `perte` (loss), `fuite` (leakage), `decoupage` (split), `groupe` (group), `etiquette` (label), `liens` (links), `mots_suspects` (suspicious words), `mediatheque` (media library), `librairie` (bookshop), `clients` / `commandes` (customers / orders), `spectateurs` / `reservations` (audience members / bookings).

In [ ]:
# Chapter 6 data, copied from the book's CSV files (do not modify)

# D6.1 — the association's three sets: messages, spams, and filter errors for a threshold
lots = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.2 — two splits of the same messages: the test set, at threshold 0.7
fuite = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['decoupage', 'messages', 'spams', 'copies_vues', 'manques_copies', 'manques_nouveaux', 'fp'])

# D6.3 — six new messages (validation for lesson 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D5.1 — the twelve messages of chapter 5 (training for lesson 6.3)
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D6.E2 — the media library (exercise 6.2): validation and test sets
mediatheque = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.4 — the association's three folds (lesson 6.4): links (mean, standard deviation) and validation losses
plis = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['pli', 'messages', 'spams', 'moyenne_liens', 'ecart_type_liens', 'perte_l001', 'perte_l01'])

# D6.5 — the bookshop's A/B test (lessons 6.5 and 6.6)
librairie = pd.DataFrame([
    ('A', 'lettre actuelle', 2500, 200),
    ('B', 'livraison offerte', 2400, 240)
], columns=['version', 'description', 'clients', 'commandes'])

# D6.E1 — the media library's cross-validation (exercise 6.1): loss of each fold for five λ
plis_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['pli', 'lambda', 'perte'])

# D6.E3 — how four groups are spread over the volunteer's folds (exercise 6.3)
groupes_plis = pd.DataFrame([
    ('Montres', 'campagne de spams', 9, 8, 9),
    ('Colis en attente', 'campagne de spams', 5, 6, 4),
    ('Loterie', 'campagne de spams', 7, 7, 8),
    ('Lettre de la mairie', 'expéditeur légitime', 4, 3, 5)
], columns=['groupe', 'nature', 'pli_1', 'pli_2', 'pli_3'])

# D6.E5 — the theatre's A/B test (exercise 6.5)
theatre = pd.DataFrame([
    ('A', 'lettre habituelle', 1800, 126),
    ('B', 'lettre avec vidéo', 1750, 147)
], columns=['version', 'description', 'spectateurs', 'reservations'])

## The calculations of the lessons, in Python

These cells redo the calculations of lessons 6.1 to 6.6.

In [ ]:
# Lesson 6.1 — cost per message (equation 6.1), k = 3 for the association
k = 3
t = lots.copy()
t['coût'] = t['fn'] + k * t['fp']
t['coût par message'] = (t['coût'] / t['messages']).round(3)    # divide by the size OF THE SET
print(t.to_string(index=False))
val = t[t['lot'] == 'validation']
choisi = val.loc[val['coût par message'].idxmin(), 'seuil']
print('threshold chosen on validation:', choisi)
test = t[(t['lot'] == 'test') & (t['seuil'] == choisi)].iloc[0]
print(f"test, only once: cost {test['coût']} for {test['messages']} messages = {test['coût par message']:.3f} per message")

a7 = t[t['seuil'] == 0.7]
couleurs = {'entrainement': LOT_E, 'validation': LOT_V, 'test': LOT_T}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['lot'], a7['coût par message'], color=[couleurs[l] for l in a7['lot']])
plt.ylabel('Cost per message (threshold 0.7)'); plt.show()

In [ ]:
# Lesson 6.2 — data leakage: recall and cost per message of the two splits
for _, l in fuite.iterrows():
    manques = l['manques_copies'] + l['manques_nouveaux']
    nouveaux = l['spams'] - l['copies_vues']
    print(f"{l['decoupage']:>12} : recall = {(l['spams'] - manques) / l['spams']:.2f}, "
          f"cost per message = {(manques + 3 * l['fp']) / l['messages']:.3f}, "
          f"share of new spams missed = {l['manques_nouveaux']} / {nouveaux} = {l['manques_nouveaux'] / nouveaux:.2f}")

# Split the campaigns, not the messages: a small demonstration with 40 campaigns of varying size
rng = np.random.default_rng(6)
campagne = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # campaign number of each spam
au_hasard = rng.random(campagne.size) < 0.25                              # 25 % of the MESSAGES in the test
camp_test = rng.choice(40, size=10, replace=False)                         # 25 % of the CAMPAIGNS in the test
par_campagne = np.isin(campagne, camp_test)
for nom, test_ in (('at random', au_hasard), ('by campaign', par_campagne)):
    partagees = len(set(campagne[test_]) & set(campagne[~test_]))
    print(f'{nom:>12} : {test_.sum()} spams in the test, {partagees} campaigns present both in the test and elsewhere')

In [ ]:
# Lesson 6.3 — overfitting: filters A and B on training (twelve messages) and validation (six)
X12, y12 = douze[['liens', 'mots_suspects']].to_numpy(float), (douze['etiquette'] == 'spam').to_numpy(int)
X6, y6 = six[['liens', 'mots_suspects']].to_numpy(float), (six['etiquette'] == 'spam').to_numpy(int)
filtres = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for nom, (w, b) in filtres.items():
    p6 = sigmoide(X6 @ np.array(w) + b)
    pertes6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filter {nom}: training {perte_moyenne(y12, X12, w, b):.2f}, validation {perte_moyenne(y6, X6, w, b):.2f} '
          f'| losses of the six: {np.round(pertes6, 2)}')

# Penalised L2 loss (equation 6.2), λ = 0.05; the constant b is not penalised
lam = 0.05
for nom, (w, b) in filtres.items():
    pen = lam * np.sum(np.square(w))
    print(f'filter {nom}: sum of squared weights = {np.sum(np.square(w)):.4f}, penalty = {pen:.4f}, '
          f'penalised loss = {perte_moyenne(y12, X12, w, b) + pen:.2f}')

# The penalty depends on the units: links counted in tens, same predictions, different penalty
X12_diz = X12 / np.array([10, 1])
w_diz = (10, 0.5)
print('same probabilities:', np.allclose(sigmoide(X12_diz @ np.array(w_diz) - 4), sigmoide(X12 @ np.array([1, 0.5]) - 4)),
      f'| penalty in units: {lam * (1 + 0.25):.4f}; in tens: {lam * (100 + 0.25):.4f}')

In [ ]:
# Lesson 6.4 — standardisation in each round: mean and standard deviation of the links over the TWO OTHER folds
for k in range(3):
    autres = plis[plis['pli'] != k + 1]
    n = autres['messages'].sum()
    m = (autres['messages'] * autres['moyenne_liens']).sum() / n
    # standard deviation of two folds combined (division by n): mean of the squares minus the square of the mean
    v = (autres['messages'] * (autres['ecart_type_liens'] ** 2 + autres['moyenne_liens'] ** 2)).sum() / n - m ** 2
    print(f'round {k + 1}: validation = fold {k + 1}, training = {n} messages, mean {m:.2f}, standard deviation {np.sqrt(v):.2f}')
print(f'message with 6 links in fold 3, in round 3: (6 - 2.25) / 1.76 = {(6 - 2.25) / 1.76:.2f}')

# Cross-validation score (equation 6.4) of the two values of λ
for col, lam in (('perte_l001', 0.01), ('perte_l01', 0.1)):
    print(f'λ = {lam}: losses of the folds {list(plis[col])}, CV = {plis[col].mean():.3f}')

In [ ]:
# Lesson 6.4 (end) — L1 and L2 on fictitious messages with eight features (the same as in exercise 6.4)
msgs = fabriquer_messages()
X8 = msgs[NOMS_CARAC].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # here, standardisation over all messages: just a demonstration
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = ajuster(X8, y8, 0.1)
w_l1, _ = ajuster(X8, y8, 0.1, l1=True)
print('L2 weights:', np.round(w_l2, 2))
print('L1 weights:', np.round(w_l1, 2))
print(f'zero weights: L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(the first two features are the only ones related to the label)')

In [ ]:
# Lessons 6.5 and 6.6 — the bookshop's A/B test
a, b_ = librairie.iloc[0], librairie.iloc[1]
pA, pB = a['commandes'] / a['clients'], b_['commandes'] / b_['clients']
ecart = pB - pA
print(f'rate A = {100 * pA:.1f} %, rate B = {100 * pB:.1f} %, difference = {100 * ecart:.1f} points, '
      f'relative change = {100 * ecart / pA:.0f} %')
esA, esB = np.sqrt(pA * (1 - pA) / a['clients']), np.sqrt(pB * (1 - pB) / b_['clients'])
es = np.sqrt(esA ** 2 + esB ** 2)                          # equation 6.6: we add the SQUARES
bas, haut = ecart - 1.96 * es, ecart + 1.96 * es           # equation 6.7
print(f'SE of A = {100 * esA:.2f} point, SE of B = {100 * esB:.2f} point, SE of the difference = {100 * es:.2f} point')
print(f'95 % confidence interval: from {100 * bas:.1f} to {100 * haut:.1f} points')
print(f'for 40 000 subscribers: {40000 * ecart:.0f} more orders (from {40000 * bas:.0f} to {40000 * haut:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0: no effect')
plt.axvline(1, color=SEUIL, linestyle='--', label='break-even threshold: 1 point')
plt.plot([100 * bas, 100 * haut], [0, 0], color=ECART, linewidth=4, label='95 % interval')
plt.scatter([100 * ecart], [0], marker='D', s=70, color=ECART, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Effect of version B (percentage points)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Lesson 6.6 — the test's cost per message (lesson 6.1) is a mean: same reasoning
couts = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 missed spams, 4 false alarms, 226 at no cost
es_c = couts.std() / np.sqrt(len(couts))                     # standard deviation (division by n) / square root of n
print(f'cost per message = {couts.mean():.3f}, standard deviation = {couts.std():.3f}, SE = {es_c:.3f}, '
      f'interval: from {couts.mean() - 1.96 * es_c:.2f} to {couts.mean() + 1.96 * es_c:.2f}')

## Exercise 6.1 — Reading a graph: three folds, five values of λ

*Concepts: cross-validation (lesson 6.4, equation 6.4), overfitting and penalty (lesson 6.3).* Data D6.E1: the validation loss of each fold of the media library, for five values of $\lambda$.

### a) to e) On paper, reading the figure in the book

*Your readings and your answers:* …

In [ ]:
# TO COMPLETE: mean of the three folds for each λ (equation 6.4), then the chosen λ
moyennes = None   # TO COMPLETE: a Series indexed by λ; try plis_lambda.groupby('lambda')['perte']
lam_choisi = None   # TO COMPLETE

In [ ]:
# Check of a) and b)
if moyennes is None or lam_choisi is None:
    print('not yet completed. Complete the two lines.')
else:
    print('means:', moyennes.round(3).to_dict(), '→',
          'correct' if np.allclose(moyennes.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'check again')
    print('chosen λ:', lam_choisi, '→', 'correct' if lam_choisi == 0.1 else 'check again')
    print('choice of each fold on its own:', plis_lambda.loc[plis_lambda.groupby('pli')['perte'].idxmin(), ['pli', 'lambda']].values.tolist())
    for k, (style, marque) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = plis_lambda[plis_lambda['pli'] == k + 1]
        plt.plot(d['lambda'], d['perte'], style, marker=marque, color=LOT_V, label=f'fold {k + 1}')
    plt.plot(moyennes.index, moyennes.values, 'D-', color='black', linewidth=2.5, label='mean')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Validation loss'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Exercise 6.2 — Choosing, then evaluating a threshold: the media library

*Concepts: three sets (lesson 6.1), cost per message (equation 6.1), cost of a threshold (equation 5.9).* For the media library, one false alarm costs as much as two missed spams: $k = 2$. Data D6.E2. First do questions a) to e) on paper, **without looking at the test rows before question b)**.

### a) to e) On paper

*Your calculations and your answers:* …

In [ ]:
# TO COMPLETE: cost per message (equation 6.1) and choice of the threshold on the VALIDATION set
def cout_par_message(fn, fp, n, k):
    return None   # TO COMPLETE


def choisir_seuil(table_validation, k):
    """Returns the threshold whose cost per message is the smallest on the validation set."""
    return None   # TO COMPLETE

In [ ]:
# Check of a) and b), then the protocol
k_med = 2
valid = mediatheque[mediatheque['lot'] == 'validation']
test_med = mediatheque[mediatheque['lot'] == 'test']
if cout_par_message(3, 18, 150, 2) is None or choisir_seuil(valid, k_med) is None:
    print('not yet completed. Complete the functions.')
else:
    cv = [cout_par_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validation, cost per message:', np.round(cv, 3), '→',
          'correct' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'check again')
    s = choisir_seuil(valid, k_med)
    print(f'threshold chosen on validation: {s} →', 'correct' if s == 0.6 else 'check again')
    l = test_med[test_med['seuil'] == s].iloc[0]                    # ONLY ONE row of the test
    ct = cout_par_message(l.fn, l.fp, l.messages, k_med)
    print(f'test, at threshold {s} only: {ct:.3f} per message →', 'correct' if abs(ct - 35 / 240) < 1e-9 else 'check again')
    # e) What the colleague does: he looks at ALL the rows of the test and keeps the best one
    ctous = [cout_par_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['seuil'].iloc[int(np.argmin(ctous))]
    print(f'what the colleague announces: threshold {s_coll}, {min(ctous):.3f} per message (chosen on the test: too optimistic);',
          f'honest announcement: threshold {s}, {ct:.3f} per message →',
          'correct' if s_coll == 0.7 and abs(min(ctous) - 28 / 240) < 1e-9 else 'check again')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['seuil'], cv, 'o-', color=LOT_V, label='validation (used to choose)')
    plt.plot(test_med['seuil'], ctous, 's--', color=LOT_T, alpha=0.5, label='test (not to be looked at when choosing)')
    plt.scatter([s], [ct], s=120, color=LOT_T, zorder=3, label='test, at the chosen threshold')
    plt.xlabel('Threshold'); plt.ylabel('Cost per message (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Your conclusion

*What the media library can announce:* …

## Exercise 6.3 — Interpreting: a leak between folds

*Concepts: data leakage (lesson 6.2), group cross-validation and standardisation in each round (lesson 6.4).* Data D6.E3: where the messages of four groups ended up in the volunteer's folds.

### a) to d) On paper

*Your answers:* …

In [ ]:
# TO COMPLETE: groups present in more than one fold
def groupes_coupes(repartition):
    """repartition: a table with a 'groupe' column and the columns 'pli_1', 'pli_2', 'pli_3'
    (number of messages of the group in each fold). Returns the list of split groups."""
    return None   # TO COMPLETE: for each group, count the folds where it has at least one message

In [ ]:
# Check of b) and d)
proposition = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['groupe', 'pli_1', 'pli_2', 'pli_3'])
if groupes_coupes(groupes_plis) is None:
    print('not yet completed. Complete the function.')
else:
    print('folds of the volunteer, split groups:', groupes_coupes(groupes_plis), '→',
          'correct' if len(groupes_coupes(groupes_plis)) == 4 else 'check again')
    print('corrected distribution, split groups:', groupes_coupes(proposition), '| fold sizes:',
          proposition[['pli_1', 'pli_2', 'pli_3']].sum().tolist(), '→',
          'correct' if groupes_coupes(proposition) == [] else 'check again')
    print('round 1: copies of « Montres » seen in training =', int(groupes_plis.loc[0, ['pli_2', 'pli_3']].sum()),
          ', in validation =', int(groupes_plis.loc[0, 'pli_1']))

## Exercise 6.4 — Cross-validation by campaign

*Concepts: cross-validation (lesson 6.4, equation 6.4), L2 penalty (equation 6.2), standardisation (equation 2.7), leakage (lesson 6.2).* The messages are made by `fabriquer_messages()` (import cell). Groups 30 to 39 form the test set: we touch it only in question e).

In [ ]:
# The messages of the exercise: 30 groups for cross-validation, 10 groups set aside for the test
msgs = fabriquer_messages()
test_ = msgs['groupe'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, NOMS_CARAC].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'groupe'].to_numpy()
X_test, y_test = msgs.loc[test_, NOMS_CARAC].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} messages, {msgs.groupe.nunique()} groups; cross-validation: {len(y_cv)} messages; test: {len(y_test)} messages')
print(msgs[NOMS_CARAC].describe().loc[['mean', 'std']].round(1))     # very different units

### a) Your prediction

*When λ goes from 0 to 1, the cross-validation loss…* …

In [ ]:
# TO COMPLETE: b) folds by group, standardisation on training, c) cross-validation score
def plis_par_groupe(groupes, K):
    """Fold number (0 to K - 1) of each message: the groups, sorted in increasing order, are dealt
    in turn into the folds; all the messages of a group get the same fold."""
    return None   # TO COMPLETE; np.unique(groupes) gives the sorted groups


def standardiser(X_entr, X_val):
    """Standardises X_entr and X_val with the mean and standard deviation of X_entr ONLY."""
    return None, None   # TO COMPLETE


def score_vc(lam, plis_, K=3):
    """Equation 6.4: mean, over the K rounds, of the loss WITHOUT penalty on the validation fold."""
    return None   # TO COMPLETE: use standardiser, ajuster(X, y, lam) and perte_moyenne(y, X, w, b)

In [ ]:
# Check of b) and c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = plis_par_groupe(g_cv, 3)
if p_g is None or standardiser(X_cv, X_cv)[0] is None or score_vc(0.1, p_g) is None:
    print('not yet completed. Complete the three functions.')
else:
    coupes = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'folds by group: {np.bincount(p_g).tolist()} messages, {coupes} split group(s) →',
          'correct' if coupes == 0 and set(p_g) == {0, 1, 2} else 'check again')
    Xe, Xv = standardiser(X_cv[p_g != 0], X_cv[p_g == 0])
    print('standardised training: means ≈ 0 and standard deviations ≈ 1; validation: not exactly →',
          'correct' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'check again')
    vc_groupe = {lam: score_vc(lam, p_g) for lam in LAMBDAS}
    lam_vc = min(vc_groupe, key=vc_groupe.get)
    print('CV by group:', {l: round(v, 3) for l, v in vc_groupe.items()}, '→ chosen λ:', lam_vc, '→',
          'correct' if lam_vc == 0.01 else 'check again')

In [ ]:
# d) Folds drawn message by message (not to be done): the leakage of lesson 6.2, three times over
if p_g is not None and score_vc(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    vc_message = {lam: score_vc(lam, p_m) for lam in LAMBDAS}
    print('CV message by message:', {l: round(v, 3) for l, v in vc_message.items()})
    plt.plot(range(5), list(vc_groupe.values()), 'o-', color=LOT_V, label='folds by group')
    plt.plot(range(5), list(vc_message.values()), 's--', color='grey', label='folds message by message (leakage)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Cross-validation score')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Final model: standardisation and weights on the 30 groups, with the chosen λ; test ONLY ONCE
    Xc, Xt = standardiser(X_cv, X_test)
    w_f, b_f = ajuster(Xc, y_cv, lam_vc)
    print(f'test loss of the final model (λ = {lam_vc}): {perte_moyenne(y_test, Xt, w_f, b_f):.3f}; '
          f'CV by group: {vc_groupe[lam_vc]:.3f}; CV message by message: {vc_message[lam_vc]:.3f}')

### d) and e) Your interpretation

…

## Exercise 6.5 — Predict, then check: the theatre's reminder letter

*Concepts: difference in points (equation 6.5), standard error (equation 6.6), confidence interval (equation 6.7).* Data D6.E5. The video is only worthwhile if it gains at least 0.5 points.

### a) to e) On paper

*Your calculations, your prediction and your answers:* …

In [ ]:
# TO COMPLETE: difference in points and 95 % confidence interval (equations 6.5 to 6.7)
def intervalle(conv_A, n_A, conv_B, n_B):
    """Returns (difference, lower bound, upper bound), in percentage points."""
    return None   # TO COMPLETE

In [ ]:
# Check of a), b) and e)
A, B = theatre.iloc[0], theatre.iloc[1]
r = intervalle(A['reservations'], A['spectateurs'], B['reservations'], B['spectateurs'])
if r is None:
    print('not yet completed. Complete the function.')
else:
    print(f'actual trial: difference {r[0]:.2f} points, interval from {r[1]:.2f} to {r[2]:.2f} →',
          'correct' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'check again')
    r4 = intervalle(4 * A['reservations'], 4 * A['spectateurs'], 4 * B['reservations'], 4 * B['spectateurs'])
    print(f'trial four times larger: from {r4[1]:.2f} to {r4[2]:.2f}; width {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correct' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'check again')

In [ ]:
# f) 1.000 simulated trials whose true effect is 1.4 points: how many intervals contain 1.4?
rng = np.random.default_rng(65)
nA, nB, vrai_A, vrai_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, vrai_A, 1000) / nA
b = rng.binomial(nB, vrai_B, 1000) / nB
es_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contient = np.abs((b - a) - 0.014) <= 1.96 * es_sim
print(f'share of intervals that contain the true effect: {contient.mean():.1%}')
print(f'share of intervals that contain zero: {np.mean(np.abs(b - a) <= 1.96 * es_sim):.1%}')

### f) Your conclusion

…

## Exercise 6.6 — Challenge: how many customers, and when to look?

*Concepts: standard error and confidence interval (lesson 6.6), binomial distribution (lesson 4.5).*

### a) to c) On paper

*Your calculations and your prediction:* …

In [ ]:
# TO COMPLETE: a) size per group for a given half-width (in points)
def taille_par_groupe(pA, pB, demi_largeur):
    """pA, pB in percent; returns the smallest integer n such that 1.96 × SE ≤ demi_largeur."""
    return None   # TO COMPLETE; np.ceil rounds up to the next integer

In [ ]:
# Check of a) and b)
n1 = taille_par_groupe(8, 10, 1)
if n1 is None:
    print('not yet completed. Complete the function.')
else:
    print(f'half-width 1 point: {n1} customers per group →', 'correct' if n1 == 6285 else 'check again')
    print(f'half-width 0.5 point: {taille_par_groupe(8, 10, 0.5)} customers per group, '
          f'{2 * taille_par_groupe(8, 10, 0.5)} in total')

In [ ]:
# d) 2.000 trials WITHOUT any effect. Schedule: 250 customers per day and per group, 20 readings (one each evening,
#    on the cumulative customers: 250, 500… 5.000 per group). Stopping rule: first evening on which the interval excludes 0.
rng = np.random.default_rng(66)
essais, jours, par_jour, p = 2000, 20, 250, 0.08
cA = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # cumulative orders, group A
cB = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # group B: same rate, no effect
n = par_jour * np.arange(1, jours + 1)                                  # customers per group at the end of each day
pA_, pB_ = cA / n, cB / n
es_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
exclut = np.abs(pB_ - pA_) > 1.96 * es_j                                # does the evening's interval exclude zero?
fin, un_soir = exclut[:, -1].mean(), exclut.any(axis=1).mean()
print(f'interval read only on day 20: {fin:.1%} of the trials wrongly conclude there is an effect')
print(f'interval read every evening, stopping at the first « success »: {un_soir:.1%}')

### d) and e) Your explanation

…